以下を行う。
```
Gaussian distribution
        ↓
   Generative Model
        ↓
digit image
```

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from torch.utils.data import DataLoader, TensorDataset

# -----------------------
# setup
# -----------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
np.random.seed(0)

# -----------------------
# data: sklearn digits
# -----------------------
digits = load_digits()
x = digits.data.astype("float32") / 16.0      # [0,1]
x = x * 2.0 - 1.0                             # [-1,1]

x_tensor = torch.tensor(x)
loader = DataLoader(TensorDataset(x_tensor), batch_size=256, shuffle=True)

data_dim = 64

# -----------------------
# diffusion schedule
# -----------------------
T = 100

betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

# -----------------------
# timestep embedding
# -----------------------
class TimeEmbedding(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.dim = dim

    def forward(self, k):
        half = self.dim // 2
        freqs = torch.exp(
            -np.log(10000) * torch.arange(half, device=k.device) / half
        )
        args = k[:, None].float() * freqs[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=1)

# -----------------------
# noise prediction model epsilon_theta(x_k, k)
# -----------------------
class DiffusionModel(nn.Module):
    def __init__(self, data_dim=64, time_dim=64, hidden=256):
        super().__init__()
        self.time_emb = TimeEmbedding(time_dim)

        self.net = nn.Sequential(
            nn.Linear(data_dim + time_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, data_dim),
        )

    def forward(self, x_k, k):
        emb = self.time_emb(k)
        return self.net(torch.cat([x_k, emb], dim=1))

model = DiffusionModel(data_dim=data_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# -----------------------
# forward diffusion q(x_k | x_0)
# x_k = sqrt(alpha_bar_k) x_0 + sqrt(1-alpha_bar_k) eps
# -----------------------
def q_sample(x0, k, eps):
    alpha_bar_k = alpha_bars[k].view(-1, 1)
    return (
        torch.sqrt(alpha_bar_k) * x0
        + torch.sqrt(1.0 - alpha_bar_k) * eps
    )

# -----------------------
# training
# -----------------------
n_epochs = 500
losses = []

for epoch in range(n_epochs):
    total_loss = 0.0

    for (x0,) in loader:
        x0 = x0.to(device)
        batch_size = x0.shape[0]

        k = torch.randint(0, T, (batch_size,), device=device)
        eps = torch.randn_like(x0)

        x_k = q_sample(x0, k, eps)
        eps_pred = model(x_k, k)

        loss = ((eps_pred - eps) ** 2).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    losses.append(total_loss / len(loader))

    if epoch % 50 == 0:
        print(f"epoch {epoch:4d} | loss = {losses[-1]:.6f}")

# -----------------------
# reverse diffusion sampling
# -----------------------
@torch.no_grad()
def sample_digits(model, n_samples=64):
    model.eval()

    x = torch.randn(n_samples, data_dim, device=device)

    for k in reversed(range(T)):
        k_batch = torch.full((n_samples,), k, device=device, dtype=torch.long)

        beta_k = betas[k]
        alpha_k = alphas[k]
        alpha_bar_k = alpha_bars[k]

        eps_pred = model(x, k_batch)

        mean = (
            1.0 / torch.sqrt(alpha_k)
            * (
                x
                - beta_k / torch.sqrt(1.0 - alpha_bar_k) * eps_pred
            )
        )

        if k > 0:
            eps = torch.randn_like(x)
            x = mean + torch.sqrt(beta_k) * eps
        else:
            x = mean

    x = (x + 1.0) / 2.0
    x = torch.clamp(x, 0, 1)
    return x.cpu().numpy()

generated = sample_digits(model, n_samples=64)

# -----------------------
# plot generated digits
# -----------------------
fig, axes = plt.subplots(8, 8, figsize=(6, 6))

for i, ax in enumerate(axes.ravel()):
    ax.imshow(generated[i].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    ax.axis("off")

plt.suptitle("Generated digits by DDPM diffusion model")
plt.tight_layout()
plt.show()

# -----------------------
# plot loss
# -----------------------
plt.figure(figsize=(7, 3))
plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Diffusion model training loss")
plt.tight_layout()
plt.show()